# Chia shot (TransNetV2 + SliTraNet)

- **Đầu vào:** video `*.mp4` (dataset Kaggle), trọng số TransNetV2, tuỳ chọn trọng số SliTraNet.
- **Đầu ra:** `/kaggle/working/scenes/<video_id>.scenes.csv` với các cột
  `video_id, scene_id, start_frame, end_frame, start_time_s, end_time_s, duration_s, fps`.
  Đây đúng là tên file mà `generate_manifest.py`, `build_object_index.py` và notebook ASR đọc.
- **Sau đó:** chép vào `data/scenes/` trên máy build.

TransNetV2 đề xuất các điểm cắt. Khi các frame trong shot biến động mạnh (camera chuyển động,
slide, đèn flash), SliTraNet kiểm tra lại và bỏ các điểm cắt mà nó thấy là video liên tục.

In [ ]:
!pip install -q ffmpeg-python opencv-python-headless

## Cấu hình

In [ ]:
from pathlib import Path

VIDEOS_DIR = Path("/kaggle/input/datasets/dwctien/aic26-videos-batch2")
TRANSNET_WEIGHTS = Path(
    "/kaggle/input/datasets/trncqun/transnetv2-weights/weights/transnetv2-weights"
)
# None để chỉ dùng TransNetV2
SLITRANET_WEIGHTS = Path(
    "/kaggle/input/datasets/trncqun/transnetv2-weights/weights/"
    "Slide_transition_detection_3DResNet50.pth"
)
OUTPUT_DIR = Path("/kaggle/working/scenes")

SHARD_INDEX = 1  # ← đổi theo máy: 1, 2, 3 hoặc 4
NUM_SHARDS = 4

CUT_THRESHOLD = 0.3  # ngưỡng điểm cắt của TransNetV2
VARIANCE_THRESHOLD = 2.0  # shot biến động hơn ngưỡng này mới hỏi SliTraNet

## TransNetV2

In [ ]:
import csv
from functools import partial

import numpy as np
import tensorflow as tf


def select_shard(items, shard_index, num_shards, weight=None):
    """Part shard_index (1-based) of num_shards. Items are sorted first so every machine agrees.

    Without weight: contiguous chunks. With weight: the heaviest item goes to the lightest
    shard (e.g. by video size), so all machines finish at about the same time.
    """
    assert 1 <= shard_index <= num_shards, "SHARD_INDEX phải nằm trong [1, NUM_SHARDS]"
    ordered = sorted(items)
    if num_shards == 1:
        return ordered
    if weight is None:
        size = -(-len(ordered) // num_shards)
        return ordered[(shard_index - 1) * size : shard_index * size]
    shards = [[] for _ in range(num_shards)]
    totals = [0.0] * num_shards
    for item in sorted(ordered, key=weight, reverse=True):
        lightest = totals.index(min(totals))
        shards[lightest].append(item)
        totals[lightest] += weight(item)
    return sorted(shards[shard_index - 1])


def matches_any(video_id, selectors):
    """True when no selector is given or video_id equals / starts with one of them."""
    return not selectors or any(video_id == s or video_id.startswith(s) for s in selectors)


class TransNetV2:
    INPUT_SIZE = (27, 48, 3)

    def __init__(self, model_dir):
        if not model_dir.is_dir():
            raise FileNotFoundError(f"Không thấy trọng số TransNetV2 ở {model_dir}")
        self._model = tf.saved_model.load(str(model_dir))

    def predict_raw(self, frames):
        logits, extra = self._model(tf.cast(frames, tf.float32))
        return tf.sigmoid(logits), tf.sigmoid(extra["many_hot"])

    def predict_frames(self, frames):
        assert frames.ndim == 4 and frames.shape[1:] == self.INPUT_SIZE

        def windows():
            # 25 frame đệm ở đầu, đủ frame ở cuối để lấp kín các bước 50 frame.
            pad_end = 25 + 50 - (len(frames) % 50 if len(frames) % 50 != 0 else 50)
            padded = np.concatenate([frames[:1]] * 25 + [frames] + [frames[-1:]] * pad_end, axis=0)
            for start in range(0, len(padded) - 99, 50):
                yield padded[start : start + 100][np.newaxis]

        single, many = [], []
        for window in windows():
            single_pred, many_pred = self.predict_raw(window)
            single.append(single_pred.numpy()[0, 25:75, 0])
            many.append(many_pred.numpy()[0, 25:75, 0])
        return np.concatenate(single)[: len(frames)], np.concatenate(many)[: len(frames)]

    def predict_video(self, video_path):
        import ffmpeg

        stream, _ = (
            ffmpeg.input(str(video_path))
            .output("pipe:", format="rawvideo", pix_fmt="rgb24", s="48x27")
            .run(capture_stdout=True, capture_stderr=True)
        )
        video = np.frombuffer(stream, np.uint8).reshape([-1, 27, 48, 3])
        return (video, *self.predict_frames(video))

    @staticmethod
    def predictions_to_scenes(predictions, threshold=0.5):
        binary = (predictions > threshold).astype(np.uint8)
        scenes = []
        t, t_prev, start = -1, 0, 0
        for i, t in enumerate(binary):
            if t_prev == 1 and t == 0:
                start = i
            if t_prev == 0 and t == 1 and i != 0:
                scenes.append([start, i])
            t_prev = t
        if t == 0:
            scenes.append([start, len(binary) - 1])
        if not scenes:
            return np.array([[0, len(binary) - 1]], dtype=np.int32)
        return np.array(scenes, dtype=np.int32)

## SliTraNet (3D ResNet-50 phân loại chuyển cảnh)

In [ ]:
import cv2
import torch
from torch import nn
from torch.nn import functional as F


def downsample_basic_block(x, planes, stride):
    out = F.avg_pool3d(x, kernel_size=1, stride=stride)
    zero_pads = out.new_zeros(
        out.size(0), planes - out.size(1), out.size(2), out.size(3), out.size(4)
    )
    return torch.cat([out, zero_pads], dim=1)


class Bottleneck3D(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.conv2 = nn.Conv3d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)
        self.conv3 = nn.Conv3d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm3d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return self.relu(out + residual)


class ResNet3DBackbone(nn.Module):
    def __init__(self, layers, shortcut_type="B"):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv3d(
            3, 64, kernel_size=7, stride=(1, 2, 2), padding=(3, 3, 3), bias=False
        )
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=1)
        self.layer1 = self._make_layer(64, layers[0], shortcut_type)
        self.layer2 = self._make_layer(128, layers[1], shortcut_type, stride=2)
        self.layer3 = self._make_layer(256, layers[2], shortcut_type, stride=2)
        self.layer4 = self._make_layer(512, layers[3], shortcut_type, stride=2)
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))

    def _make_layer(self, planes, blocks, shortcut_type, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * Bottleneck3D.expansion:
            if shortcut_type == "A":
                downsample = partial(
                    downsample_basic_block, planes=planes * Bottleneck3D.expansion, stride=stride
                )
            else:
                downsample = nn.Sequential(
                    nn.Conv3d(
                        self.inplanes,
                        planes * Bottleneck3D.expansion,
                        kernel_size=1,
                        stride=stride,
                        bias=False,
                    ),
                    nn.BatchNorm3d(planes * Bottleneck3D.expansion),
                )
        layers = [Bottleneck3D(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * Bottleneck3D.expansion
        layers += [Bottleneck3D(self.inplanes, planes) for _ in range(1, blocks)]
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.avgpool(x)


class SliTraNet(nn.Module):
    def __init__(self, n_class=4):
        super().__init__()
        self.backbone_3d = ResNet3DBackbone([3, 4, 6, 3])
        self.conv_final = nn.Conv2d(2048, n_class, kernel_size=1, bias=False)

    def forward(self, x):
        return self.conv_final(torch.squeeze(self.backbone_3d(x), dim=2))


def load_slitranet(weights, device):
    net = SliTraNet()
    trained = torch.load(weights, map_location="cpu", weights_only=True)
    state = net.state_dict()
    for key in state:
        pretrained = trained.get(f"module.{key}")
        if pretrained is not None and pretrained.size() == state[key].size():
            state[key] = pretrained
        elif pretrained is not None:
            print(f"[WARN] SliTraNet lệch kích thước ở {key}")
    net.load_state_dict(state, strict=True)
    return net.to(device).eval()

## Chia shot

In [ ]:
CONTINUOUS_VIDEO_CLASS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transnet = TransNetV2(TRANSNET_WEIGHTS)
slitranet = load_slitranet(SLITRANET_WEIGHTS, DEVICE) if SLITRANET_WEIGHTS else None


def shot_variance(video_frames, start, end):
    """Std of mean frame differences, sampled every 5 frames of the 48x27 thumbnails."""
    frames = video_frames[start:end:5]
    if end - start < 2 or len(frames) < 2:
        return 0.0
    gray = np.dot(frames[..., :3], [0.2989, 0.5870, 0.1140])
    return float(np.std(np.mean(np.abs(gray[1:] - gray[:-1]), axis=(1, 2))))


def is_real_cut(video_path, boundary_frame):
    if slitranet is None:
        return True
    clip_length, patch_size = 8, 256
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, boundary_frame - clip_length // 2))
    frames = []
    for _ in range(clip_length):
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if len(frames) < clip_length:
        return True
    clip = np.stack(
        [cv2.resize(f, (patch_size, patch_size), interpolation=cv2.INTER_NEAREST) for f in frames]
    )
    tensor = torch.from_numpy(clip).permute(3, 0, 1, 2).float() / 255.0 - 0.5
    with torch.inference_mode():
        logits = slitranet(tensor.unsqueeze(0).to(DEVICE))
        predicted = int(logits.squeeze(2).squeeze(2).softmax(dim=1).argmax(dim=1).item())
    return predicted != CONTINUOUS_VIDEO_CLASS


def detect_scenes(video_path):
    video_frames, single_pass, _ = transnet.predict_video(video_path)
    raw_scenes = transnet.predictions_to_scenes(single_pass, threshold=CUT_THRESHOLD)
    scenes = []
    current_start = int(raw_scenes[0][0])
    for i, (_, end) in enumerate(raw_scenes):
        last = i == len(raw_scenes) - 1
        if not last:
            variance = shot_variance(video_frames, current_start, int(end))
            if variance > VARIANCE_THRESHOLD and not is_real_cut(video_path, int(end)):
                continue  # SliTraNet thấy video liên tục: bỏ điểm cắt sai
        scenes.append((current_start, int(end)))
        if not last:
            current_start = int(raw_scenes[i + 1][0])
    return scenes


def video_fps(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps if fps > 0 else 25.0


def write_scenes(csv_path, video_id, scenes, fps):
    columns = [
        "video_id", "scene_id", "start_frame", "end_frame",
        "start_time_s", "end_time_s", "duration_s", "fps",
    ]  # fmt: skip
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for scene_id, (start, end) in enumerate(scenes):
            writer.writerow(
                {
                    "video_id": video_id,
                    "scene_id": scene_id,
                    "start_frame": start,
                    "end_frame": end,
                    "start_time_s": start / fps,
                    "end_time_s": end / fps,
                    "duration_s": (end - start) / fps,
                    "fps": fps,
                }
            )


from tqdm.auto import tqdm  # noqa: E402

videos = select_shard(list(VIDEOS_DIR.rglob("*.mp4")), SHARD_INDEX, NUM_SHARDS)
print(f"Shard {SHARD_INDEX}/{NUM_SHARDS}: {len(videos)} video, SliTraNet: {slitranet is not None}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
failed = {}
for video_path in tqdm(videos, desc="Video"):
    csv_path = OUTPUT_DIR / f"{video_path.stem}.scenes.csv"
    if csv_path.exists():
        continue  # chạy lại cell để tiếp tục sau khi bị ngắt
    try:
        scenes = detect_scenes(video_path)
    except Exception as exc:  # noqa: BLE001 - một video lỗi không được dừng cả lô
        failed[video_path.stem] = str(exc)
        continue
    write_scenes(csv_path, video_path.stem, scenes, video_fps(video_path))

print(f"Xong {len(videos) - len(failed)}/{len(videos)} video")
for video_id, error in failed.items():
    print(f"[LỖI] {video_id}: {error}")

## Nén kết quả
Tải file nén ở tab **Output** của Kaggle rồi giải nén vào `data/scenes/` trên máy build.

In [ ]:
import shutil

archive = shutil.make_archive(
    str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_shard{SHARD_INDEX}of{NUM_SHARDS}"),
    "zip",
    root_dir=str(OUTPUT_DIR.parent),
    base_dir=OUTPUT_DIR.name,
)
print(f"{archive}: {Path(archive).stat().st_size / 2**20:.1f} MB")